In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
import requests

url = "https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt"
text = requests.get(url).text.lower()

# 🔥 LIMIT DATA
text = text[:200000]   # only first 200k chars

print("Length:", len(text))

Length: 200000


In [3]:
chars = sorted(list(set(text)))

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

vocab_size = len(chars)

data = torch.tensor([stoi[c] for c in text])

print("Vocab:", vocab_size)

Vocab: 61


In [4]:
seq_len = 50

def get_batch(data, batch_size=64):
    ix = torch.randint(len(data)-seq_len, (batch_size,))
    X = torch.stack([data[i:i+seq_len] for i in ix])
    Y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    return X, Y

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)

        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0)/d_model))

        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)

        self.pe = pe.unsqueeze(0)

    def forward(self,x):
        return x + self.pe[:,:x.size(1)]

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.heads = heads
        self.dk = d_model // heads

        self.q = nn.Linear(d_model,d_model)
        self.k = nn.Linear(d_model,d_model)
        self.v = nn.Linear(d_model,d_model)

        self.fc = nn.Linear(d_model,d_model)

    def forward(self,x):
        B,T,C = x.shape

        q = self.q(x).view(B,T,self.heads,self.dk)
        k = self.k(x).view(B,T,self.heads,self.dk)
        v = self.v(x).view(B,T,self.heads,self.dk)

        scores = torch.matmul(q,k.transpose(-2,-1))/np.sqrt(self.dk)
        attn = torch.softmax(scores,dim=-1)

        out = torch.matmul(attn,v)
        out = out.contiguous().view(B,T,C)

        return self.fc(out)

In [7]:
class Block(nn.Module):
    def __init__(self,d_model,heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model,heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model,4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model,d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self,x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

In [8]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.embed = nn.Embedding(vocab_size,64)
        self.pos = PositionalEncoding(64)

        self.blocks = nn.Sequential(
            Block(64,2),
            Block(64,2)
        )

        self.fc = nn.Linear(64,vocab_size)

    def forward(self,x):
        x = self.embed(x)
        x = self.pos(x)
        x = self.blocks(x)
        return self.fc(x)

In [11]:
model = Transformer()
opt = optim.Adam(model.parameters(),0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(20):
    Xb, Yb = get_batch(data)

    out = model(Xb)
    loss = loss_fn(out.view(-1,vocab_size), Yb.view(-1))

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("Epoch",epoch+1,"Loss:",loss.item())

Epoch 1 Loss: 4.191684722900391
Epoch 2 Loss: 3.941354751586914
Epoch 3 Loss: 3.7273759841918945
Epoch 4 Loss: 3.552994966506958
Epoch 5 Loss: 3.3951215744018555
Epoch 6 Loss: 3.368617057800293
Epoch 7 Loss: 3.327126979827881
Epoch 8 Loss: 3.2637741565704346
Epoch 9 Loss: 3.201432228088379
Epoch 10 Loss: 3.162264108657837
Epoch 11 Loss: 3.0977272987365723
Epoch 12 Loss: 3.0896823406219482
Epoch 13 Loss: 3.06368350982666
Epoch 14 Loss: 3.0408074855804443
Epoch 15 Loss: 2.9936211109161377
Epoch 16 Loss: 2.973317861557007
Epoch 17 Loss: 2.9588332176208496
Epoch 18 Loss: 2.956430435180664
Epoch 19 Loss: 2.862602472305298
Epoch 20 Loss: 2.906771183013916


In [12]:
import torch.nn.functional as F

def generate(start, length=200, temperature=0.8):
    model.eval()
    x = torch.tensor([[stoi[c] for c in start]])

    for _ in range(length):
        out = model(x)
        probs = F.softmax(out[:,-1]/temperature, dim=-1)

        next_char = torch.multinomial(probs, 1).item()

        x = torch.cat([x, torch.tensor([[next_char]])], dim=1)

    return ''.join([itos[i] for i in x[0].tolist()])

print(generate("the king "))

the king  
w 
  t  ie    -ny nyrfii  uo
f  l  oea
  lf oanr    c s  fer  a o?r s ) t lscaald 
   _ a wb y(ahoo iia
eyi terha eo2 ads) fc fjenpilhes  euin>a tc -i o]hcrhe toyl    t
 th)nhe 2wasoii inreten t1an 
